# Limpieza y preparación del conjunto de datos

## 1. Introducción

En esta actividad se realiza la **limpieza y preparación** del conjunto de datos *Online Shoppers Purchasing Intention Dataset*.

La limpieza de datos es un paso fundamental antes de cualquier análisis o modelado. Su objetivo es asegurar que los datos sean **consistentes, completos y válidos** para que los resultados posteriores sean confiables.

A partir de la exploración realizada en el notebook anterior (`01_revision_datos.ipynb`), se identificaron los siguientes aspectos a corregir:

1. **Registros duplicados**: 125 filas idénticas en todas sus columnas.
2. **Inconsistencia en nombres de meses**: la variable `Month` tiene formatos mixtos (ej. `June` vs `Jun`, `Jul` vs `Jul`).
3. **Valores atípicos**: aunque no son errores, se documentan para decisión posterior.

El resultado de esta limpieza se guardará como un archivo CSV procesado que servirá como base para el diseño del Data Mart (Unidad II), los modelos supervisados (Unidad III) y el clustering (Unidad IV).

## 2. Importación de bibliotecas y carga de datos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Detectar raíz del proyecto de forma robusta
raiz = Path.cwd()
while raiz.name and raiz.name != 'REC_ECDB_2026_ChavezMolina_Byron':
    if raiz.parent == raiz:
        break
    raiz = raiz.parent

archivo_original = raiz / 'data' / 'raw' / 'online_shoppers_intention.csv'
print(f"Archivo existe: {archivo_original.exists()}")

datos = pd.read_csv(archivo_original)
print(f"Dimensión original: {datos.shape}")
datos.head()

## 3. Diagnóstico previo a la limpieza

Antes de aplicar cualquier transformación, confirmamos los hallazgos de la exploración.

In [ ]:
# 3.1 Valores faltantes
faltantes = datos.isnull().sum()
print("=== Valores faltantes ===")
print(faltantes[faltantes > 0] if faltantes.any() else "No hay valores faltantes")

# 3.2 Registros duplicados exactos
duplicados = datos.duplicated().sum()
print(f"\n=== Registros duplicados exactos: {duplicados} ===")

# 3.3 Valores únicos en Month (inconsistencia detectada)
print(f"\n=== Meses únicos antes de limpieza: {sorted(datos['Month'].unique())} ===")

# 3.4 Tipos de visitante
print(f"Tipos de visitante: {datos['VisitorType'].unique()}")

# 3.5 Variables objetivo
print(f"Revenue únicos: {datos['Revenue'].unique()}")
print(f"Weekend únicos: {datos['Weekend'].unique()}")

## 4. Proceso de limpieza

Aplicamos las transformaciones necesarias paso a paso, explicando **qué** se hace y **por qué**.

### 4.1 Eliminación de registros duplicados

Los **registros duplicados exactos** son filas que tienen **idénticos valores en las 18 columnas**. En un contexto de sesiones de navegación, esto puede ocurrir por errores de captura o registro doble.

**Decisión**: se eliminan manteniendo solo la primera ocurrencia (`keep='first'`). Esto reduce el ruido sin perder información única.

*Nota técnica*: `drop_duplicates()` por defecto compara todas las columnas. Como no hay clave primaria en el CSV crudo, esta es la forma correcta de detectar duplicados completos.

In [ ]:
filas_antes = len(datos)
datos_limpios = datos.drop_duplicates().copy()
filas_despues = len(datos_limpios)
eliminados = filas_antes - filas_despues

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Duplicados eliminados: {eliminados}")

### 4.2 Estandarización de nombres de mes

La variable `Month` almacena el mes de la visita como **cadena de texto** (tipo categórico nominal). En la exploración se observó que:

- La mayoría usa abreviatura de **3 letras**: `Feb`, `Mar`, `May`, `Oct`, `Jul`, `Aug`, `Nov`, `Sep`, `Dec`
- Dos meses usan formato distinto: **`June`** (4 letras) en lugar de `Jun`

Esta inconsistencia **rompe la uniformidad** de la categoría y causaría problemas al:
- Agrupar por mes (contaría `June` y `Jun` como meses distintos)
- Crear la dimensión `DimMes` en el Data Mart
- Visualizaciones ordenadas cronológicamente

**Decisión**: normalizar `June` → `Jun` para que todos los meses tengan formato de 3 letras.

*Nota técnica*: No se detectaron otros formatos inconsistentes (ej. `Jul` ya está correcto con 3 letras).

In [ ]:
# Verificar valores antes
print(f"Meses antes: {sorted(datos_limpios['Month'].unique())}")

# Normalizar: June -> Jun
datos_limpios['Month'] = datos_limpios['Month'].replace({'June': 'Jun'})

# Verificar después
print(f"Meses después: {sorted(datos_limpios['Month'].unique())}")

# Confirmar que solo cambió June
conteo_meses = datos_limpios['Month'].value_counts().sort_index()
print("\nConteo por mes:")
print(conteo_meses)

### 4.3 Verificación de valores negativos (validación de dominio)

Aunque la exploración no mostró negativos, es **buena práctica** validar explícitamente las variables donde un valor negativo **no tiene sentido semántico**:

- Conteos de páginas: `Administrative`, `Informational`, `ProductRelated`
- Duraciones (segundos): `Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration`
- Tasas y proporciones: `BounceRates`, `ExitRates` (rango [0,1])
- Valores monetarios/índices: `PageValues`, `SpecialDay`

Si existieran, indicarían **error de captura o transformación previa**.

In [ ]:
variables_no_negativas = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates',
    'PageValues', 'SpecialDay'
]

negativos = (datos_limpios[variables_no_negativas] < 0).sum()
print("=== Valores negativos por variable ===")
print(negativos[negativos > 0] if negativos.any() else "No hay valores negativos en variables de dominio no negativo")

### 4.4 Verificación de categorías válidas

Confirmamos que las variables categóricas solo contienen los valores esperados.

In [ ]:
print("=== Valores únicos por variable categórica ===")
print(f"VisitorType: {sorted(datos_limpios['VisitorType'].unique())}")
print(f"Month: {sorted(datos_limpios['Month'].unique())}")
print(f"Weekend: {sorted(datos_limpios['Weekend'].unique())}")
print(f"Revenue: {sorted(datos_limpios['Revenue'].unique())}")
print(f"OperatingSystems: {sorted(datos_limpios['OperatingSystems'].unique())}")
print(f"Browser: {sorted(datos_limpios['Browser'].unique())}")
print(f"Region: {sorted(datos_limpios['Region'].unique())}")
print(f"TrafficType: {sorted(datos_limpios['TrafficType'].unique())}")

## 5. Resumen de la limpieza

Consolidamos los cambios realizados en un diccionario de resumen.

In [ ]:
resumen_limpieza = {
    "filas_originales": filas_antes,
    "duplicados_eliminados": eliminados,
    "filas_finales": filas_despues,
    "valores_faltantes_finales": int(datos_limpios.isnull().sum().sum()),
    "meses_estandarizados": "June -> Jun",
    "valores_negativos_encontrados": 0
}

print("=== RESUMEN DE LIMPIEZA ===")
for clave, valor in resumen_limpieza.items():
    print(f"  {clave}: {valor}")

## 6. Guardado del dataset limpio

El dataset limpio se guarda en la carpeta `data/processed/clean/` para ser utilizado por:

- **ETL hacia SQL Server** (`src/etl/cargar_sql.py`)
- **Modelos supervisados** (Unidad III)
- **Clustering y PCA** (Unidad IV)
- **Dashboard** (`dashboard/app.py`)

In [ ]:
# Ruta de salida
archivo_salida = raiz / 'data' / 'processed' / 'clean' / 'online_shoppers_intention_clean.csv'

# Crear directorio si no existe
archivo_salida.parent.mkdir(parents=True, exist_ok=True)

# Guardar sin índice
datos_limpios.to_csv(archivo_salida, index=False)

print(f"Archivo guardado en: {archivo_salida}")
print(f"Tamaño: {archivo_salida.stat().st_size / 1024:.1f} KB")
print(f"Filas: {len(datos_limpios):,}")
print(f"Columnas: {len(datos_limpios.columns)}")

## 7. Verificación final del archivo guardado

In [ ]:
# Leer y verificar
verificacion = pd.read_csv(archivo_salida)

print(f"Dimensión: {verificacion.shape}")
print(f"Valores faltantes: {verificacion.isnull().sum().sum()}")
print(f"Duplicados: {verificacion.duplicated().sum()}")
print(f"Meses únicos: {sorted(verificacion['Month'].unique())}")
print(f"\nTipos de dato:")
print(verificacion.dtypes)
print(f"\nPrimeras 5 filas:")
verificacion.head()

## 8. Conclusiones

La limpieza del conjunto de datos **Online Shoppers Purchasing Intention Dataset** se completó con las siguientes acciones:

| Acción | Detalle | Impacto |
|--------|---------|---------|
| **Eliminación de duplicados** | 125 filas idénticas removidas | Reduce sesgo en conteos y modelos |
| **Estandarización de meses** | `June` → `Jun` (formato 3 letras) | Consistencia categórica para agrupaciones y Data Mart |
| **Validación de dominio** | 0 valores negativos en variables de conteo/duración/tasas | Confirma integridad semántica |
| **Validación de categorías** | Todos los valores dentro del dominio esperado | Sin categorías espurias |

### Resultado final
- **Registros**: 12,205 filas (antes 12,330)
- **Variables**: 18 columnas sin cambios
- **Calidad**: Sin valores faltantes, sin duplicados, categorías consistentes

El archivo `online_shoppers_intention_clean.csv` está listo para:
1. **Carga al Data Mart** (esquema estrella en SQL Server)
2. **Entrenamiento de modelos** de regresión, clasificación y clustering
3. **Visualización** en el dashboard interactivo

---

> **Nota para el profesor**: La limpieza se realizó de forma **conservadora** (solo correcciones objetivas: duplicados exactos e inconsistencia de formato documentada). No se eliminaron valores atípicos ni se imputaron valores, ya que el dataset original no presenta faltantes y los atípicos representan comportamientos reales de navegación que enriquecen el análisis.